# The Agent Loop, Explained

> **Companion notebook** to [The AI agent loop: the architecture powering autonomous AI](BLOG_URL) — Oracle Developer Blog

---

Imagine you ask an AI assistant: *"Find the three cheapest flights to Tokyo next month, check if my loyalty points cover any of them, and book the best option."*

A standard chatbot fails this task immediately. It can answer the first part, but then it stops — waiting for your next message, having already forgotten the context. It is a bit like a shop assistant who answers your question and then immediately forgets you exist. Useful for one-off queries; useless for anything that requires follow-through.

The reason is architectural. A single-pass chatbot has three hard limitations:

1. **It cannot act.** It generates text; it cannot call APIs, run code, or change state in external systems.
2. **It cannot adapt.** There is no feedback loop. If the first approach fails, it cannot try a different one.
3. **It cannot decompose.** Multi-step tasks require breaking a problem into sub-problems, solving each in order, and accumulating results.

The **agent loop** is the architectural fix for all three. It is deceptively simple:

```python
while not done:
    response = call_llm(messages)
    if response has tool_calls:
        results = execute_tools(response.tool_calls)
        messages.append(results)
    else:
        done = True
        return response
```

An LLM calling tools inside a `while` loop, iterating until the task is complete. Every major AI agent — from OpenAI's Agents SDK to Google's Deep Research to Anthropic's Claude Code — is built on this same pattern.

This notebook makes that loop **visible and concrete**. Using three simple tools — a calculator, a unit converter, and a timezone converter — you will watch the loop execute step by step. Oracle AI Database 26ai stores every iteration so you can query your agent's full execution history with SQL.

## What You Will Learn

1. What the five stages of the agent loop are and how they map to real code
2. How LangChain's agent framework implements the `while` loop pattern step by step
3. How to define tools that an agent can call and observe
4. How to make the loop visible — streaming every step of reasoning and tool use live
5. How to store and retrieve agent observations as vectors in Oracle AI Database 26ai using semantic search

## Prerequisites

| Requirement | Details |
|-------------|---------|
| Python | 3.10 or higher |
| Podman or Docker | To run Oracle AI Database 26ai locally (free container image) |
| LLM (choose one path) | **Path A — Local:** Ollama running on your machine (no API key needed). **Path B/C — API:** An OpenAI or Google Gemini API key. |
| Python packages | Installed in the next cell |

> **Path A (Ollama)** runs entirely on your machine with no cloud accounts required. **Paths B and C (OpenAI / Gemini)** require an API key but need no local model runtime. All three paths produce the same output. The configuration cell below explains exactly what to put in your `.env` for each.

## What Is the Agent Loop?

### Single-turn chatbot vs. agent loop

A traditional LLM interaction is **stateless and single-turn**:

```
User  ──► LLM  ──► Response
          (done)
```

An agent loop is **iterative and stateful**:

```
                  ┌─────────────────────────────────────────────┐
                  │               AGENT LOOP                    │
                  │                                             │
User Task ──────► │  PERCEIVE ──► REASON ──► ACT ──► OBSERVE   │
                  │     ▲                              │        │
                  │     └──────────────────────────────┘        │
                  │           (repeat until done)               │
                  └─────────────────────────────────────────────┘
                                    │
                               Final Answer
```

### The five stages

| Stage | What happens | In this notebook |
|-------|-------------|------------------|
| **Perceive** | The agent receives input from its environment | User's question + tool results from previous iterations |
| **Reason** | The LLM decides what to do next | The LLM reads the conversation and chooses a tool |
| **Plan** | The agent optionally decomposes the task | Done implicitly by the LLM in the system prompt |
| **Act** | The agent executes a tool | `calculate`, `convert_units`, or `timezone_convert` runs |
| **Observe** | The agent reads the result and loops | Tool output goes back into the message history |

### Why this matters

The loop is what enables an agent to:
- Break a complex problem into steps it can solve one at a time
- Recover from errors by trying a different approach
- Accumulate information across multiple tool calls
- Stop when it has a complete answer — not before

As Anthropic's engineering team wrote: *"Agents are typically just LLMs using tools based on environmental feedback in a loop."* The sophistication is in the tools and the quality of reasoning — not in the loop itself.

### The research behind the pattern

In 2022, researchers at Google and Princeton formalised this loop in a paper called **ReAct** (Reasoning + Acting). By interleaving chain-of-thought reasoning with tool use — rather than separating them — ReAct-style agents improved task completion by **34% on ALFWorld** benchmarks and **10% on WebShop** compared to reasoning-only approaches. The key insight: letting the model *reason about what to do* and *act to get new information* in the same loop creates a virtuous cycle where each iteration makes the next one smarter.

Since then, every major AI platform has converged on the same core structure: OpenAI's Agents SDK, Anthropic's Claude, Google's Gemini agents, Microsoft's Semantic Kernel, and LangChain all implement the same PERCEIVE → REASON → ACT → OBSERVE cycle under different names. The loop is settled science. What varies is the scaffolding around it — tools, memory, observability, and multi-agent coordination.

In [ ]:
# Install required packages
# ─────────────────────────────────────────────────────────────────────────────
# After this cell finishes, Jupyter may restart the kernel automatically.
# If it does, you MUST re-run ALL cells from the top before continuing.
# The restart is normal — it ensures the newly installed packages are loaded.

%pip install -q \
    oracledb \
    "langchain>=0.3" \
    langchain-oracledb \
    langchain-ollama \
    langchain-openai \
    langchain-google-genai \
    python-dotenv

print("Packages installed.")
print("If the kernel restarted, re-run all cells from alf-05-config before continuing.")

## Configuration

Before running the next cell, create a `.env` file in the same directory as this notebook. Choose one of the three paths below — you only need the variables for your chosen path.

---

### Path A: Ollama (local, no API key needed)

Run your own models on your machine with [Ollama](https://ollama.com/download). No cloud account required.

Copy the following into your `.env` file:

```
ORACLE_USER=system
ORACLE_PASSWORD=YourPassword123
ORACLE_DSN=localhost:1521/FREE

OLLAMA_MODEL=llama3.2:3b
OLLAMA_EMBED_MODEL=nomic-embed-text
```

`LLM_PROVIDER` defaults to `"ollama"` if not set, so you do not need to add it explicitly. The Ollama server is assumed to be running at `http://localhost:11434`.

---

### Path B: OpenAI API

Use GPT models via the OpenAI API. Obtain a key at [platform.openai.com](https://platform.openai.com/api-keys).

Copy the following into your `.env` file:

```
ORACLE_USER=system
ORACLE_PASSWORD=YourPassword123
ORACLE_DSN=localhost:1521/FREE

LLM_PROVIDER=openai
LLM_MODEL=gpt-5-mini-2025-08-07
OPENAI_API_KEY=sk-...
```

---

### Path C: Google Gemini API

Use Gemini models via the Google AI API. Obtain a key at [aistudio.google.com](https://aistudio.google.com/apikey).

Copy the following into your `.env` file:

```
ORACLE_USER=system
ORACLE_PASSWORD=YourPassword123
ORACLE_DSN=localhost:1521/FREE

LLM_PROVIDER=gemini
LLM_MODEL=gemini-3-flash-preview
GOOGLE_API_KEY=AIza...
```

---

### All environment variables

| Variable | Required for | Default | Purpose |
|---|---|---|---|
| `ORACLE_USER` | All paths | `system` | Database username |
| `ORACLE_PASSWORD` | All paths | `YourPassword123` | Database password |
| `ORACLE_DSN` | All paths | `localhost:1521/FREE` | Database connection string |
| `LLM_PROVIDER` | All paths | `ollama` | Which LLM backend to use: `ollama`, `openai`, or `gemini` |
| `OLLAMA_MODEL` | Path A only | `llama3.2:3b` | Chat model served by Ollama |
| `OLLAMA_EMBED_MODEL` | Path A only | `nomic-embed-text` | Embedding model served by Ollama |
| `LLM_MODEL` | Paths B and C | — | Model name passed to the API (e.g. `gpt-5-mini-2025-08-07` or `gemini-3-flash-preview`) |
| `OPENAI_API_KEY` | Path B only | _(none)_ | Your OpenAI API key |
| `GOOGLE_API_KEY` | Path C only | _(none)_ | Your Google AI API key |

In [ ]:
import os
from dotenv import load_dotenv

load_dotenv(override=True)  # override=True ensures .env values win over shell env vars

# ── LLM Provider ──────────────────────────────────────────────────────────────
# Supported values: "ollama" (default, local), "openai", "gemini"
LLM_PROVIDER = os.getenv("LLM_PROVIDER", "ollama")

# ── Ollama (local — no API key needed) ────────────────────────────────────────
OLLAMA_MODEL       = os.getenv("OLLAMA_MODEL",       "llama3.2:3b")
OLLAMA_EMBED_MODEL = os.getenv("OLLAMA_EMBED_MODEL", "nomic-embed-text")

# ── OpenAI / Gemini (API key providers) ──────────────────────────────────────
# LLM_MODEL must be set in .env for your chosen provider:
#   OpenAI  → gpt-5-mini-2025-08-07
#   Gemini  → gemini-3-flash-preview
LLM_MODEL      = os.getenv("LLM_MODEL",      "")
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GOOGLE_API_KEY = os.getenv("GOOGLE_API_KEY", "")

# ── Oracle AI Database 26ai ───────────────────────────────────────────────────
ORACLE_USER     = os.getenv("ORACLE_USER",     "system")
ORACLE_PASSWORD = os.getenv("ORACLE_PASSWORD", "YourPassword123")
ORACLE_DSN      = os.getenv("ORACLE_DSN",      "localhost:1521/FREE")

print(f"LLM provider      : {LLM_PROVIDER}")
if LLM_PROVIDER == "ollama":
    print(f"Ollama model      : {OLLAMA_MODEL}")
    print(f"Ollama embed model: {OLLAMA_EMBED_MODEL}")
else:
    if not LLM_MODEL:
        print("WARNING: LLM_MODEL is not set. Add it to your .env file.")
        print("  OpenAI example : LLM_MODEL=gpt-5-mini-2025-08-07")
        print("  Gemini example : LLM_MODEL=gemini-3-flash-preview")
    else:
        print(f"LLM model         : {LLM_MODEL}")
print(f"Oracle DSN        : {ORACLE_DSN}")
print(f"Oracle user       : {ORACLE_USER}")

## Ollama Setup

Ollama runs Llama 3.2 and the embedding model entirely on your local machine. No API key, no cloud account, no data leaving your network.

**Step 1: Install Ollama**

Download the installer for your platform from [ollama.com/download](https://ollama.com/download).

**Step 2: Pull the models**

Open a terminal and run:

```bash
ollama pull llama3.2:3b
ollama pull nomic-embed-text
```

The first command pulls Llama 3.2 3B (approximately 2 GB). The second pulls the `nomic-embed-text` embedding model (approximately 270 MB). Both are stored in `~/.ollama/models`.

**Step 3: Start the Ollama server**

```bash
ollama serve
```

The server listens on `http://localhost:11434` by default. You can verify it is running:

```bash
curl http://localhost:11434
# Should return: Ollama is running
```

> **Why Llama 3.2 3B?** Llama 3.2 3B is Meta's compact, instruction-tuned model built specifically for edge and local deployment. At 3 billion parameters it runs comfortably on a laptop CPU with 8 GB of RAM. It supports tool calling natively, which is what the agent loop depends on, and is available directly in Ollama's official library. For a more capable agent (at the cost of more RAM), replace `llama3.2:3b` with `llama3.2:8b` or `mistral:7b` in your `.env` file.

> **Why `nomic-embed-text` for embeddings?** `nomic-embed-text` is a well-regarded open-source embedding model with a 768-dimensional output. It produces meaningful semantic embeddings for short tool observation strings, is available in Ollama's standard library, and its dimension count is a sensible default for Oracle Vector Search. Alternatively, `mxbai-embed-large` produces 1024-dimensional embeddings for higher precision, at the cost of a slightly larger download (670 MB).

## Oracle AI Database 26ai Setup

Oracle AI Database 26ai is Oracle's latest database release with native AI capabilities built in, including vector search, JSON Duality Views, and SQL Property Graph. In this notebook we use it to store agent observations as vectors, enabling semantic retrieval over past tool calls via Oracle Vector Search.

**Start the database with Podman (Oracle Linux / RHEL):**

```bash
podman run -d \
  --name oracle-26ai \
  -p 1521:1521 \
  -e ORACLE_PWD=YourPassword123 \
  container-registry.oracle.com/database/free:latest
```

**Or with Docker (macOS / Windows / Ubuntu):**

```bash
docker run -d \
  --name oracle-26ai \
  -p 1521:1521 \
  -e ORACLE_PWD=YourPassword123 \
  container-registry.oracle.com/database/free:latest
```

> **Image tag:** `:latest` always pulls the current release, which is fine for getting started. For a reproducible environment, replace `:latest` with a specific version tag (e.g. `23.7.0.0`). Available tags are listed at [container-registry.oracle.com](https://container-registry.oracle.com/ords/ocr/ba/database/free).

> **ORACLE_PWD must match ORACLE_PASSWORD.** The `ORACLE_PWD` value you pass to the container sets the password for the `system` account. The `ORACLE_PASSWORD` variable in alf-05-config must be identical — a mismatch causes `ORA-01017: invalid username/password`. If you changed one, change both.

> The database takes approximately 60 seconds to initialise on first launch. Monitor progress with `podman logs -f oracle-26ai` (or `docker logs -f oracle-26ai`). Proceed once you see `DATABASE IS READY TO USE!`

> **Privileges:** The connection uses the `system` account by default, which has `CREATE TABLE` permission. If you switch to a custom schema user, grant `CREATE TABLE` and `INSERT` to that user before running the next cell.

In [ ]:
import oracledb

# ── Connection pool ───────────────────────────────────────────────────────────
# A pool rather than a single connection means later cells can acquire and
# release connections independently without holding one open the whole time.
# OracleVS accepts a ConnectionPool directly — the pool is passed in at
# vector store creation time rather than acquired per operation.
pool = oracledb.create_pool(
    user=ORACLE_USER,
    password=ORACLE_PASSWORD,
    dsn=ORACLE_DSN,
    min=2,
    max=10,
    increment=1,
)

# ── Default tablespace ────────────────────────────────────────────────────────
# The SYSTEM tablespace uses manual segment space management, which blocks
# JSON and VECTOR column types (ORA-43853). Switching to USERS — which uses
# automatic segment space management — allows OracleVS to create its table.
with pool.acquire() as conn:
    with conn.cursor() as cur:
        cur.execute(f"ALTER USER {ORACLE_USER} DEFAULT TABLESPACE USERS")
    conn.commit()
print(f"Default tablespace for '{ORACLE_USER}' set to USERS.")

# ── Smoke test ────────────────────────────────────────────────────────────────
try:
    with pool.acquire() as conn:
        with conn.cursor() as cur:
            cur.execute("SELECT 'Oracle connection OK' FROM dual")
            row = cur.fetchone()
    print(f"Connected to Oracle AI Database 26ai: {row[0]}")
    print("Connection pool ready (min=2, max=10).")
except oracledb.DatabaseError as e:
    error, = e.args
    if error.code == 1017:
        print("ERROR ORA-01017: invalid username/password.")
        print("Check that ORACLE_PASSWORD in alf-05-config matches the")
        print("ORACLE_PWD value used when starting the container.")
    elif error.code == 12541:
        print("ERROR ORA-12541: no listener. Is the container running?")
    else:
        raise

In [ ]:
# ── LLM initialisation ────────────────────────────────────────────────────────
# Default is Ollama (local, no API key needed). To switch providers, set
# LLM_PROVIDER in .env to "openai" or "gemini" and supply the matching key.
# temperature=0 keeps the agent deterministic across all providers.

if LLM_PROVIDER == "openai":
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model=LLM_MODEL, api_key=OPENAI_API_KEY, temperature=0)

elif LLM_PROVIDER == "gemini":
    from langchain_google_genai import ChatGoogleGenerativeAI
    llm = ChatGoogleGenerativeAI(model=LLM_MODEL, google_api_key=GOOGLE_API_KEY, temperature=0)

else:  # ollama — default, runs locally on http://localhost:11434
    from langchain_ollama import ChatOllama
    llm = ChatOllama(model=OLLAMA_MODEL, temperature=0)

# ── Smoke test ────────────────────────────────────────────────────────────────
# Confirms the chosen provider is reachable and the model is loaded.
# Gemini returns content as a list of parts; other providers return a plain
# string. Normalise to a string before printing.
test = llm.invoke("Reply with exactly three words: agent loop ready")
reply = (
    test.content
    if isinstance(test.content, str)
    else "".join(
        p.get("text", str(p)) if isinstance(p, dict) else str(p)
        for p in test.content
    )
)
print(f"LLM check    : {reply.strip()}")
print(f"Provider     : {LLM_PROVIDER}")

## The Agent's Tools

Tools are the mechanism by which the agent **acts** on the world. In the `ACT` stage of the loop, the LLM decides which tool to call and with what arguments. The result goes back into the message history as an `OBSERVE` event — the agent reads it and decides what to do next.

We define three tools. They are intentionally simple — pure Python, no external APIs, no authentication — so the **loop mechanics** are the focus, not the tools themselves.

### Tool 1: `calculate`

Evaluates a mathematical expression. The agent uses this for arithmetic it cannot do reliably in its head.

In [ ]:
import math
from langchain_core.tools import tool


@tool
def calculate(expression: str) -> str:
    """Evaluate a mathematical expression and return the numeric result.

    Supports standard arithmetic (+, -, *, /), powers (**), and common math
    functions: sqrt, floor, ceil, round, abs, min, max.

    Args:
        expression: A mathematical expression string, e.g. "5570 / 900" or "round(6.19 * 60, 1)"

    Returns:
        The numeric result as a string, or an error message.

    Note:
        Uses eval() with a restricted namespace. Suitable for demo purposes only —
        do not pass untrusted user input directly to this function in production.
    """
    safe_builtins = {
        name: getattr(math, name)
        for name in dir(math)
        if not name.startswith("__")
    }
    safe_builtins.update({
        "abs": abs, "round": round, "int": int, "float": float,
        "min": min, "max": max,
    })
    try:
        result = eval(expression, {"__builtins__": {}}, safe_builtins)
        if isinstance(result, float):
            return str(round(result, 6))
        return str(result)
    except Exception as e:
        return f"Error evaluating '{expression}': {e}"


# Verify the tool works before connecting it to the agent
print("calculate('5570 / 900') →", calculate.invoke({"expression": "5570 / 900"}))
print("calculate('round(6.188889 * 60, 2)') →", calculate.invoke({"expression": "round(6.188889 * 60, 2)"}))

### Tool 2: `convert_units`

Converts a value between units of distance, speed, or time. The agent calls this when the problem involves units that need to change — kilometres to miles, hours to minutes, km/h to knots, and so on.

In [ ]:
@tool
def convert_units(value: float, from_unit: str, to_unit: str) -> str:
    """Convert a numeric value from one unit to another.

    Supported unit groups:
      Distance : km, miles, meters, feet, nautical miles
      Speed    : km/h, mph, m/s, knots
      Time     : hours, minutes, seconds

    Units within the same group can be converted freely.
    Cross-group conversions (e.g. km to hours) will return an error.

    Args:
        value     : The numeric value to convert
        from_unit : Source unit, e.g. "km/h", "miles", "hours"
        to_unit   : Target unit, e.g. "mph", "km",   "minutes"

    Returns:
        The converted value with target unit label, or an error message.
    """
    # Conversion factors relative to base unit per group
    # (distance base = km, speed base = km/h, time base = hours)
    to_base = {
        "km": 1.0,          "miles": 1.60934,    "meters": 0.001,
        "feet": 0.0003048,  "nautical miles": 1.852,
        "km/h": 1.0,        "mph": 1.60934,      "m/s": 3.6,
        "knots": 1.852,
        "hours": 1.0,       "minutes": 1 / 60,   "seconds": 1 / 3600,
    }
    groups = {
        "distance": {"km", "miles", "meters", "feet", "nautical miles"},
        "speed":    {"km/h", "mph", "m/s", "knots"},
        "time":     {"hours", "minutes", "seconds"},
    }

    f, t = from_unit.lower().strip(), to_unit.lower().strip()

    from_grp = next((g for g, units in groups.items() if f in units), None)
    to_grp   = next((g for g, units in groups.items() if t in units), None)

    if from_grp is None:
        return f"Unknown unit '{from_unit}'. Supported: {', '.join(sorted(to_base))}"
    if to_grp is None:
        return f"Unknown unit '{to_unit}'. Supported: {', '.join(sorted(to_base))}"
    if from_grp != to_grp:
        return f"Cannot convert {from_grp} unit '{from_unit}' to {to_grp} unit '{to_unit}'."

    result = value * to_base[f] / to_base[t]
    return f"{round(result, 4)} {to_unit}"


print("convert_units(6.188889, 'hours', 'minutes') →",
      convert_units.invoke({"value": 6.188889, "from_unit": "hours", "to_unit": "minutes"}))
print("convert_units(900, 'km/h', 'mph') →",
      convert_units.invoke({"value": 900, "from_unit": "km/h", "to_unit": "mph"}))

### Tool 3: `timezone_convert`

Converts a local time from one city to another. The agent uses this when the problem spans time zones — converting a departure time to an arrival time in a different city, for example.

In [ ]:
@tool
def timezone_convert(time_str: str, from_city: str, to_city: str) -> str:
    """Convert a local time from one city's timezone to another.

    Uses standard (non-DST) UTC offsets. Supported cities include:
      UTC+0  : London, Dublin, Accra
      UTC+1  : Paris, Berlin, Amsterdam, Rome, Madrid
      UTC+2  : Cairo, Helsinki, Athens
      UTC+3  : Moscow, Riyadh, Nairobi
      UTC+4  : Dubai
      UTC+5.5: Mumbai, New Delhi, Kolkata
      UTC+8  : Singapore, Hong Kong, Beijing, Shanghai, Perth
      UTC+9  : Tokyo, Seoul
      UTC+10 : Sydney, Melbourne, Brisbane
      UTC-3  : São Paulo, Buenos Aires
      UTC-5  : New York, Toronto, Lima
      UTC-6  : Chicago, Mexico City
      UTC-8  : Los Angeles, Vancouver, Seattle
      UTC-10 : Honolulu

    Args:
        time_str  : Local time in HH:MM (24-hour format), e.g. "14:00"
        from_city : Source city name (case-insensitive)
        to_city   : Destination city name (case-insensitive)

    Returns:
        The equivalent local time in the destination city, with a day offset note
        if the conversion crosses midnight.
    """
    offsets = {
        "london": 0, "dublin": 0, "accra": 0,
        "paris": 1, "berlin": 1, "amsterdam": 1, "rome": 1, "madrid": 1,
        "cairo": 2, "helsinki": 2, "athens": 2,
        "moscow": 3, "riyadh": 3, "nairobi": 3,
        "dubai": 4,
        "mumbai": 5.5, "new delhi": 5.5, "kolkata": 5.5,
        "singapore": 8, "hong kong": 8, "beijing": 8, "shanghai": 8, "perth": 8,
        "tokyo": 9, "seoul": 9,
        "sydney": 10, "melbourne": 10, "brisbane": 10,
        "auckland": 12,
        "sao paulo": -3, "são paulo": -3, "buenos aires": -3,
        "new york": -5, "toronto": -5, "lima": -5,
        "chicago": -6, "mexico city": -6,
        "los angeles": -8, "vancouver": -8, "seattle": -8,
        "honolulu": -10,
    }

    f = from_city.lower().strip()
    t = to_city.lower().strip()

    if f not in offsets:
        return f"Unknown city '{from_city}'. See the tool docstring for supported cities."
    if t not in offsets:
        return f"Unknown city '{to_city}'. See the tool docstring for supported cities."

    try:
        h, m = map(int, time_str.strip().split(":"))
    except ValueError:
        return f"Invalid time format '{time_str}'. Use HH:MM (24-hour), e.g. '14:00'."

    total_min  = h * 60 + m
    utc_min    = total_min - int(offsets[f] * 60)
    dest_min   = utc_min   + int(offsets[t] * 60)

    day_note = ""
    if dest_min >= 1440:
        dest_min -= 1440
        day_note = " (next day)"
    elif dest_min < 0:
        dest_min += 1440
        day_note = " (previous day)"

    dh, dm = dest_min // 60, dest_min % 60
    to_off  = offsets[t]

    return (
        f"{dh:02d}:{dm:02d} local time in {to_city.title()}"
        f" (UTC{to_off:+.4g}){day_note}"
    )


print("timezone_convert('20:11', 'London', 'New York') →",
      timezone_convert.invoke({"time_str": "20:11", "from_city": "London", "to_city": "New York"}))
print("timezone_convert('14:00', 'London', 'Tokyo') →",
      timezone_convert.invoke({"time_str": "14:00", "from_city": "London", "to_city": "Tokyo"}))

## The Loop: Pseudocode vs. Reality

Before building the agent, let us map the abstract loop directly to the LangChain code that implements it.

```
PSEUDOCODE                              LANGCHAIN EQUIVALENT
──────────────────────────────────────  ────────────────────────────────────────────────
while not done:                         create_agent runs this graph internally

    response = call_llm(messages)       agent node: llm.invoke(messages + tool schema)

    if response has tool_calls:         graph routes to "tools" node if tool_calls exist

        results = execute_tools(...)        each tool is looked up by name and called

        messages.append(results)            ToolMessage results appended to state

    else:                               else:
        done = True                         graph routes to END
        return response                     final AIMessage content is the answer
```

`create_agent` (from `langchain.agents`) returns a compiled agent graph. It has two nodes: `agent` (the LLM reasoning step) and `tools` (tool execution). It routes between them until the LLM returns a message with no tool calls. The graph stops when:
- The LLM returns a response with **no tool calls** (task complete)
- The graph's `recursion_limit` is reached (pass `config={"recursion_limit": N}` to `.stream()`)
- An unrecoverable error is raised

Every time through the loop, the tool results accumulate in the graph's `messages` state. We stream this state so you can watch each iteration as it happens.

In [ ]:
# This cell shows the while loop running manually — without create_agent —
# to make the iteration structure explicit before we hand it to LangChain.

from langchain_core.messages import HumanMessage, AIMessage, ToolMessage

tools_map = {
    "calculate":        calculate,
    "convert_units":    convert_units,
    "timezone_convert": timezone_convert,
}

# Bind tools so the LLM knows what it can call
llm_with_tools = llm.bind_tools(list(tools_map.values()))

# A minimal manual loop — the same logic create_agent encapsulates
def run_loop_manually(question: str, max_iter: int = 8):
    messages = [HumanMessage(content=question)]

    for iteration in range(1, max_iter + 1):
        print(f"\n{'─' * 50}")
        print(f"Iteration {iteration}  [REASON]")

        response = llm_with_tools.invoke(messages)   # REASON stage
        messages.append(response)

        if not response.tool_calls:                  # No tools → DONE
            print(f"  No tool calls → loop exits")
            print(f"\nFinal answer: {response.content}")
            return response.content

        for call in response.tool_calls:             # ACT stage
            print(f"  [ACT]     → {call['name']}({call['args']})")
            tool_fn = tools_map[call["name"]]
            result  = tool_fn.invoke(call["args"])
            print(f"  [OBSERVE] ← {result}")
            messages.append(ToolMessage(content=str(result), tool_call_id=call["id"]))

    return "Max iterations reached."


# Run the loop on a simple warm-up question
_ = run_loop_manually("How many minutes are in 6.189 hours?")

## Building the Agent

Now we construct the agent using LangChain's `create_agent`: the recommended function for building a tool-calling agent in LangChain 0.3+. It wires the LLM, tools, and system prompt together into a compiled agent that handles the loop internally.

```python
from langchain.agents import create_agent
```

The key parameters:

| Parameter | What it does |
|---|---|
| `model` | The LLM. Must support tool binding (`bind_tools`). `ChatOllama` does. |
| `tools` | List of `@tool`-decorated functions the agent can call. |
| `system_prompt` | A string or `SystemMessage` prepended before every LLM call. |

`create_agent` returns a compiled agent object that supports `.invoke()` and `.stream()` directly — no `AgentExecutor` wrapper needed.

In [ ]:
from langchain.agents import create_agent

# ── Tool list ─────────────────────────────────────────────────────────────────
tools = [calculate, convert_units, timezone_convert]

# ── System prompt ─────────────────────────────────────────────────────────────
SYSTEM_PROMPT = """You are a precise reasoning assistant with access to three tools:
- calculate: for arithmetic and mathematical expressions
- convert_units: for converting between distance, speed, and time units
- timezone_convert: for converting times between cities

When solving a problem:
1. Break it into steps and solve each step with a tool call.
2. Show your reasoning between steps: explain what each result means and what you will do next.
3. Do not guess or approximate — use tools for all numeric calculations.
4. When you have a complete answer, state it clearly in plain language."""

# ── Agent ─────────────────────────────────────────────────────────────────────
# create_agent returns a compiled agent. The loop runs until the LLM returns
# a response with no tool calls, or the recursion_limit is reached.
agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt=SYSTEM_PROMPT,
)

print(f"Agent ready. Tools: {[t.name for t in tools]}")
print(f"Model: {OLLAMA_MODEL}")

## Run the Agent

The cell below runs the agent on a multi-step reasoning question that requires all three tools.

Watch the output stream live. You will see the loop in action:

| Output marker | Loop stage |
|---------------|------------|
| `[ACT] → tool_name(args)` | **ACT** — the LLM has decided on a tool call |
| `[OBSERVE] ← result` | **OBSERVE** — the tool result feeds back into the graph state |
| `Answer: ...` | Loop exits: the LLM returned a final response with no further tool calls |

In [ ]:
from langchain_core.messages import AIMessage, ToolMessage

QUESTION = (
    "A flight from London to New York JFK covers 5,570 km. "
    "The aircraft cruises at 900 km/h. "
    "The flight departs London at 14:00 local time. "
    "How long is the flight in hours and minutes, "
    "and what local time does it arrive in New York?"
)

print(f"Question: {QUESTION}\n")
print("=" * 60)

# ── Stream the agent ──────────────────────────────────────────────────────────
# stream_mode="values" yields the full messages list after every graph node.
# chunk["messages"][-1] is always the most recent event.

tool_calls_made = []   # List of dicts: {tool, input, result}
final_answer    = ""
pending_calls   = {}   # Maps tool_call_id → {tool, input} until result arrives

for chunk in agent.stream(
    {"messages": [("human", QUESTION)]},
    stream_mode="values",
):
    last_msg = chunk["messages"][-1]

    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        # ── REASON + ACT stage: LLM has decided on tool calls ─────────────────
        for call in last_msg.tool_calls:
            print(f"\nIteration {len(tool_calls_made) + len(pending_calls) + 1}")
            print(f"  [ACT]     → {call['name']}({call['args']})")
            pending_calls[call["id"]] = {"tool": call["name"], "input": call["args"]}

    elif isinstance(last_msg, ToolMessage):
        # ── OBSERVE stage: tool result has come back ───────────────────────────
        call_id = last_msg.tool_call_id
        if call_id in pending_calls:
            entry = pending_calls.pop(call_id)
            result_str = str(last_msg.content)
            print(f"  [OBSERVE] ← {result_str}")
            tool_calls_made.append({
                "tool":   entry["tool"],
                "input":  entry["input"],
                "result": result_str,
            })

    elif isinstance(last_msg, AIMessage) and last_msg.content:
        # ── Final answer: LLM returned content with no tool calls ──────────────
        final_answer = last_msg.content

print("\n" + "=" * 60)
print(f"\nAnswer: {final_answer}")

# ── Build observation strings for vector storage ──────────────────────────────
# Each observation is a human-readable summary of one complete tool call cycle.
# These strings are embedded and stored in OracleVS in the next section.
observation_texts = [
    f"Iteration {i}: tool={step['tool']} input={step['input']} result={step['result']}"
    for i, step in enumerate(tool_calls_made, 1)
]

print(f"\nCapturing {len(observation_texts)} observation(s) for vector storage.")

## Inspect the Loop

The verbose output above shows the loop as it ran. The cell below presents the collected tool calls as a clean iteration-by-iteration trace — every tool the agent called, what it passed in, and what came back.

This is how you understand *why* the agent took the path it did.

In [ ]:
print(f"Loop completed in {len(tool_calls_made)} iteration(s)\n")
print("=" * 60)

for i, step in enumerate(tool_calls_made, 1):
    print(f"\nIteration {i}")
    print(f"  Stage    : ACT")
    print(f"  Tool     : {step['tool']}")
    print(f"  Input    : {step['input']}")
    print(f"  Stage    : OBSERVE")
    obs = str(step["result"])
    print(f"  Result   : {obs[:200]}{'...' if len(obs) > 200 else ''}")

print(f"\n{'=' * 60}")
print(f"Total tool calls: {len(tool_calls_made)}")

## Store Agent Observations in Oracle Vector Search

This is where the notebook pivots from a plain SQL trace table to something more interesting. Rather than storing agent execution traces as rows, we embed each observation (tool call + result) and store it in Oracle's vector search engine via `OracleVS` from `langchain-oracledb`.

Why does this matter? It means we can later ask questions like:

- "Find observations where the agent converted a distance"
- "Which past agent runs involved timezone calculations?"

...and get semantically relevant results, even if the exact words do not match. This is semantic memory: retrieval by meaning rather than by keyword.

### What gets stored

Each observation is a short string describing one iteration of the loop:

```
Iteration 1: tool=calculate input={'expression': '5570 / 900'} result=6.188889
```

`OracleVS` embeds this string using `OllamaEmbeddings` (the `nomic-embed-text` model running locally via Ollama) and stores the resulting 768-dimensional vector alongside the original text in Oracle's native `VECTOR` column type.

### The table `OracleVS` creates

`OracleVS.__init__()` calls `CREATE TABLE IF NOT EXISTS` on first use. The schema is:

| Column | Type | Purpose |
|---|---|---|
| `id` | `VARCHAR2(200)` | Document ID (UUID by default) |
| `text` | `CLOB` | The original observation string |
| `metadata` | `CLOB` (JSON) | Arbitrary key-value metadata |
| `embedding` | `VECTOR(768, FLOAT32)` | The vector produced by `nomic-embed-text` |

We pass `run_id`, `iteration`, and `tool` as metadata so the SQL inspection cell can filter and audit by run.

In [ ]:
import uuid
from langchain_ollama import OllamaEmbeddings
from langchain_oracledb import OracleVS
from langchain_community.vectorstores.utils import DistanceStrategy

# ── Embedding model ───────────────────────────────────────────────────────────
# OllamaEmbeddings calls the local Ollama server to produce vector embeddings.
# nomic-embed-text produces 768-dimensional float vectors — a sensible balance
# of quality and storage cost for short observation strings.
embeddings = OllamaEmbeddings(model=OLLAMA_EMBED_MODEL)

# ── Verify the embedding model is reachable ───────────────────────────────────
test_vec = embeddings.embed_query("agent observation test")
print(f"Embedding model : {OLLAMA_EMBED_MODEL}")
print(f"Vector dimensions: {len(test_vec)}")

# ── Vector store ──────────────────────────────────────────────────────────────
# OracleVS takes the connection pool directly and creates the AGENT_OBSERVATIONS
# table on first use (DDL is wrapped in IF NOT EXISTS internally).
# COSINE works well for comparing short text fragments where direction
# matters more than magnitude.
vector_store = OracleVS(
    client=pool,
    embedding_function=embeddings,
    table_name="AGENT_OBSERVATIONS",
    distance_strategy=DistanceStrategy.COSINE,
    # OracleVS embeds this string to detect the model's output dimension on
    # first use — the actual content doesn't matter, it just needs to be a
    # representative string of the kind we will be storing.
    query="agent tool observation",
)
print("OracleVS vector store ready.")
print("Table: AGENT_OBSERVATIONS")

## Store Observations and Build Up History

Now we store the observations from the first run, then run the agent on a second question and store those observations too. With two runs in the vector store, the next section can demonstrate semantic retrieval across both of them.

This is the pattern that powers agent memory in production systems: instead of scanning every past run with `SELECT *`, the agent embeds its query and retrieves only the most semantically relevant history.

In [ ]:
# ── Store run 1 observations ──────────────────────────────────────────────────
run_id_1 = str(uuid.uuid4())

# Each observation carries metadata so we can filter or audit by run later.
metadatas_1 = [
    {
        "run_id":    run_id_1,
        "iteration": i,
        "tool":      step["tool"],
    }
    for i, step in enumerate(tool_calls_made, 1)
]

ids_1 = vector_store.add_texts(
    texts=observation_texts,
    metadatas=metadatas_1,
)

print(f"Run 1 stored: {len(ids_1)} observation(s), run_id={run_id_1[:8]}...")

# ── Run 2: marathon world record question ─────────────────────────────────────
# A different domain (distance + speed, no timezone) gives the vector store
# meaningful variety — so the semantic retrieval queries below can pull
# observations from both runs rather than everything coming from one run.
QUESTION2 = (
    "A marathon is 42.195 km. The world record is 2 hours, 0 minutes, and 35 seconds. "
    "What was the average speed in km/h and in mph?"
)

tool_calls_made2 = []
pending_calls2   = {}

for chunk in agent.stream(
    {"messages": [("human", QUESTION2)]},
    stream_mode="values",
):
    last_msg = chunk["messages"][-1]
    if isinstance(last_msg, AIMessage) and last_msg.tool_calls:
        for call in last_msg.tool_calls:
            pending_calls2[call["id"]] = {"tool": call["name"], "input": call["args"]}
    elif isinstance(last_msg, ToolMessage):
        call_id = last_msg.tool_call_id
        if call_id in pending_calls2:
            entry = pending_calls2.pop(call_id)
            tool_calls_made2.append({
                "tool":   entry["tool"],
                "input":  entry["input"],
                "result": str(last_msg.content),
            })

# ── Store run 2 observations ──────────────────────────────────────────────────
run_id_2 = str(uuid.uuid4())

observation_texts_2 = [
    f"Iteration {i}: tool={step['tool']} input={step['input']} result={step['result']}"
    for i, step in enumerate(tool_calls_made2, 1)
]
metadatas_2 = [
    {"run_id": run_id_2, "iteration": i, "tool": step["tool"]}
    for i, step in enumerate(tool_calls_made2, 1)
]

ids_2 = vector_store.add_texts(
    texts=observation_texts_2,
    metadatas=metadatas_2,
)

print(f"Run 2 stored: {len(ids_2)} observation(s), run_id={run_id_2[:8]}...")
print(f"\nTotal observations in vector store: {len(ids_1) + len(ids_2)}")

## Semantic Retrieval and SQL Inspection

With two runs stored in Oracle's vector engine, we can now query by meaning rather than by keyword. The cell below demonstrates three retrieval patterns, then shows the raw SQL to confirm what Oracle stored.

### How Oracle Vector Search works here

`similarity_search(query, k=3)` translates to this inside Oracle:

```sql
SELECT id, text, metadata
FROM agent_observations
ORDER BY VECTOR_DISTANCE(embedding, :query_vec, COSINE)
FETCH FIRST 3 ROWS ONLY
```

The query string is first embedded using `nomic-embed-text`, then Oracle computes the cosine distance between that query vector and every stored observation vector, returning the closest matches. No full-text indexing, no keyword matching — pure vector geometry.

In [ ]:
# ── Semantic retrieval demo ───────────────────────────────────────────────────

print("=" * 60)
print("Query 1: 'distance conversion'")
print("=" * 60)
results = vector_store.similarity_search("distance conversion", k=3)
for doc in results:
    meta = doc.metadata
    print(f"  run={meta.get('run_id','?')[:8]}... | iter={meta.get('iteration')} | tool={meta.get('tool')}")
    print(f"  {doc.page_content[:120]}")
    print()

print("=" * 60)
print("Query 2: 'what time does the flight arrive'")
print("=" * 60)
results2 = vector_store.similarity_search("what time does the flight arrive", k=3)
for doc in results2:
    meta = doc.metadata
    print(f"  run={meta.get('run_id','?')[:8]}... | iter={meta.get('iteration')} | tool={meta.get('tool')}")
    print(f"  {doc.page_content[:120]}")
    print()

print("=" * 60)
print("Query 3: 'speed in miles per hour'")
print("=" * 60)
results3 = vector_store.similarity_search("speed in miles per hour", k=3)
for doc in results3:
    meta = doc.metadata
    print(f"  run={meta.get('run_id','?')[:8]}... | iter={meta.get('iteration')} | tool={meta.get('tool')}")
    print(f"  {doc.page_content[:120]}")
    print()

# ── SQL inspection ────────────────────────────────────────────────────────────
# oracledb thin mode returns VARCHAR2/RAW columns as bytes. RAW values (like
# SYS_GUID-based IDs) may contain arbitrary bytes, so we hex-encode those
# and fall back to latin-1 for anything else that isn't valid UTF-8.
# This helper ensures clean string output regardless of Oracle column type.
def to_str(val):
    if val is None:
        return ""
    if isinstance(val, bytes):
        try:
            return val.decode("utf-8")
        except UnicodeDecodeError:
            # RAW/GUID bytes — show as hex so they're always printable
            return val.hex()
    return str(val)

print("=" * 60)
print("SQL inspection: AGENT_OBSERVATIONS")
print("=" * 60)

with pool.acquire() as conn:
    with conn.cursor() as cur:
        cur.execute("SELECT COUNT(*) FROM agent_observations")
        count = cur.fetchone()[0]
        print(f"Total rows: {count}")

        cur.execute(
            "SELECT id, SUBSTR(text, 1, 80), metadata "
            "FROM agent_observations "
            "ORDER BY id "
            "FETCH FIRST 5 ROWS ONLY"
        )
        print(f"\n{'ID':<38}  {'Text (truncated)':<82}  Metadata")
        print("-" * 140)
        for row in cur.fetchall():
            id_str   = to_str(row[0])[:38]
            text_str = to_str(row[1])[:82]
            meta_str = to_str(row[2])[:40]
            print(f"{id_str:<38}  {text_str:<82}  {meta_str}")

        cur.execute(
            "SELECT column_name, data_type "
            "FROM user_tab_columns "
            "WHERE table_name = 'AGENT_OBSERVATIONS'"
        )
        print("\nTable schema:")
        for row in cur.fetchall():
            print(f"  {to_str(row[0]):<20} {to_str(row[1])}")

## What Controls the Loop?

The agent loop is not infinite — it has stopping conditions. Understanding them is essential for building agents that behave predictably in production.

### Stopping conditions

| Condition | What happens | How to configure |
|-----------|-------------|------------------|
| **Goal achieved** | The LLM returns a response with no tool calls: it believes the task is complete | Achieved automatically; the quality of the system prompt affects how reliably this happens |
| **Recursion limit** | The graph has cycled `recursion_limit` times without finishing | Pass `config={"recursion_limit": N}` to `agent.stream()` — default is 25 |
| **Error escalation** | A tool raises an unrecoverable exception | Wrap tool logic in try/except and return an error string; the LLM reads it and can self-correct |
| **Token budget** | The accumulated message history fills the context window | Use a token-counting tool the agent checks before expensive calls |

### The danger of runaway loops

Without a `recursion_limit`, a confused agent can loop indefinitely — calling the same tools with the same arguments, never making progress. Always set a limit, and consider adding a token counter as a tool the agent checks before expensive operations.

### From demo to production

This notebook runs the agent loop synchronously in a notebook cell. In production you would typically:
- Run the loop **asynchronously** (`agent.astream()` / `agent.ainvoke()`)
- **Checkpoint** agent state between iterations (pass a `checkpointer=` to `create_agent`)
- **Monitor** each iteration with structured logging (OCI Logging, OpenTelemetry)
- Add **human-in-the-loop** approval for high-impact actions (`interrupt_before=["tools"]`)

Each of these patterns is documented in the Oracle AI Developer Hub notebooks.

## The Cost of the Loop

Stopping conditions are not just an academic concern — they are a cost control mechanism.

Agents consume **roughly 4x more tokens** than a standard single-turn chat interaction. Multi-agent systems, where agents spawn sub-agents, can reach **15x more tokens** than a basic chat call. For an agent running dozens of iterations across hundreds of daily tasks, this compounds quickly.

The consequences of getting this wrong can be swift. One production system called a broken tool 400 times in five minutes — accumulating thousands of tokens of tool results that went nowhere — before anyone noticed the loop was stuck. Stopping conditions prevented nothing, because none had been set.

The practical takeaway: **cost-per-task matters more than cost-per-token** when planning an agent deployment. Instrument your loop from day one. Log every iteration, track token consumption per run, and set `recursion_limit` before you go anywhere near production.

## Seeing `recursion_limit` in Action

Talk is cheap. The cell below sets `recursion_limit=2` and asks a question that requires at least three tool calls. The graph will stop mid-task and raise a `GraphRecursionError` — exactly what you want in production when an agent stalls.

In [ ]:
# The flight question needs 3 tool calls (calculate → convert_units →
# timezone_convert). With recursion_limit=2, the graph will halt after
# 2 iterations and raise a recursion error rather than looping forever.
try:
    for chunk in agent.stream(
        {"messages": [("human", QUESTION)]},
        config={"recursion_limit": 2},
        stream_mode="values",
    ):
        last_msg = chunk["messages"][-1]
        if hasattr(last_msg, "tool_calls") and last_msg.tool_calls:
            for call in last_msg.tool_calls:
                print(f"  [ACT] → {call['name']}({call['args']})")
        elif hasattr(last_msg, "content") and last_msg.content:
            print(f"  [ANSWER] {last_msg.content[:120]}")

except Exception as e:
    print(f"\nLoop halted after 2 iterations: {type(e).__name__}")
    print("In production: catch this, log the partial state, and escalate or retry.")

## Learn More

### Companion blog post
[**The AI agent loop: the architecture powering autonomous AI**](BLOG_URL) — Oracle Developer Blog  
The conceptual deep-dive behind this notebook: how OpenAI, Anthropic, Google, Microsoft, Meta, and LangChain each implement and extend the same core loop, and what the industry has converged on.

### Oracle AI Developer Hub
More notebooks on agent memory, RAG retrieval, reasoning strategies, and multi-agent architectures:  
[github.com/oracle-devrel/oracle-ai-developer-hub](https://github.com/oracle-devrel/oracle-ai-developer-hub)

### Reference documentation
- [Oracle 26ai Free container image](https://container-registry.oracle.com/ords/ocr/ba/database/free)
- [python-oracledb documentation](https://python-oracledb.readthedocs.io/)
- [langchain-oracledb — Oracle's LangChain integration](https://github.com/oracle/langchain-oracle)
- [LangChain agents documentation](https://python.langchain.com/docs/concepts/agents/)
- [Ollama model library](https://ollama.com/library)
- [Llama 3.2 on Ollama](https://ollama.com/library/llama3.2)
- [nomic-embed-text on Ollama](https://ollama.com/library/nomic-embed-text)

### Foundational reading
- [ReAct: Synergizing Reasoning and Acting in Language Models](https://arxiv.org/abs/2210.03629) — the research paper that formalised the Thought → Action → Observation loop
- [Anthropic: Building Effective Agents](https://www.anthropic.com/research/building-effective-agents) — the most-cited industry guide on agent architecture